In [14]:
from db_config import CONNECTION_DB
from tokens_and_passwords import get_db_connection
import pandas as pd
from analysis import Analysis
from datetime import datetime

In [15]:
db = CONNECTION_DB(get_db_connection()[0], get_db_connection()[1], get_db_connection()[2])
analysis = Analysis()
analysis.recommendations()

Подключение к БД установлено


СОЗДАНИЕ ТАБЛИЦ

In [16]:
db.execute("""
CREATE TABLE IF NOT EXISTS tickers (
           name VARCHAR(10) PRIMARY KEY)
""")

In [17]:
db.execute("""
CREATE TABLE IF NOT EXISTS signals (
           ticker_name VARCHAR REFERENCES tickers(name),
           start_date VARCHAR(50) NOT NULL,
           signal VARCHAR(10) NOT NULL,
           start_price FLOAT NOT NULL,
           price_now FLOAT NOT NULL,
           score FLOAT NOT NULL
           )
""")

In [18]:
db.execute("""
CREATE TABLE IF NOT EXISTS history (
           ticker_name VARCHAR REFERENCES tickers(name),
           signal VARCHAR(10) NOT NULL,
           start_date VARCHAR(50) NOT NULL,
           end_date VARCHAR(50) NOT NULL,
           start_price FLOAT NOT NULL,
           end_price FLOAT NOT NULL,
           delta FLOAT NOT NULL
           )
""")

In [19]:
db.execute("""
CREATE TABLE IF NOT EXISTS recommendations (
           name VARCHAR REFERENCES tickers(name),
           start_price FLOAT NOT NULL,
           support_line FLOAT NOT NULL,
           resistance_line FLOAT NOT NULL,
           buy_by FLOAT NOT NULL,
           sell_by FLOAT NOT NULL,
           RSI FLOAT NOT NULL,
           Z_Score FLOAT NOT NULL,
           P_E FLOAT NOT NULL,
           ROE FLOAT NOT NULL,
           div_yield FLOAT NOT NULL,
           signal VARCHAR(10) NOT NULL,
           score FLOAT NOT NULL)
""")

ОБНОВЛЯЕМ ДАННЫЕ В ТАБЛИЦЕ 'tickers'

In [20]:
tickers_in_db = pd.read_sql_query("""SELECT * FROM tickers""",con=db.conn).values
new_tickers = pd.read_csv('candles.csv').columns[1:]
tickers_records = ', '.join([f"('{ticker}')" for ticker in new_tickers if ticker not in tickers_in_db])
columns = '(name)'

if tickers_records:
    db.insert_into(table='tickers',columns=columns, values=tickers_records)

C:\Users\Danii\AppData\Local\Temp\ipykernel_13132\1110660957.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tickers_in_db = pd.read_sql_query("""SELECT * FROM tickers""",con=db.conn).values


ОБНОВЛЯЕМ ДАННЫЕ В ТАБЛИЦУ 'recommendations'

In [21]:
recommendations = analysis.get_recommendations()
recommendations_records = ', '.join([f"{tuple(recommendation)}" for recommendation in recommendations.values])
columns = '(name, start_price, support_line, resistance_line, buy_by, sell_by, RSI, Z_score, P_E, ROE, div_yield, signal, score)'

db.clear_table('recommendations')
db.insert_into(table='recommendations', columns=columns, values=recommendations_records)

ОБНОВЛЯЕМ ТАБЛИЦУ 'signals' И 'history'

In [22]:
table_of_signals = pd.read_sql_query("""SELECT * FROM signals""", db.conn).set_index('ticker_name')
signals = pd.concat([analysis.get_buy_list(), analysis.get_sell_list()])
prices = pd.read_csv('candles.csv').set_index('Date')
columns_signals = '(ticker_name, start_date, signal, start_price, price_now, score)'
columns_history = '(ticker_name, signal, start_date, end_date, start_price, end_price, delta)'

for ticker in table_of_signals.index:
    price_now = prices[ticker].sort_index().iloc[-1]
    db.update_table('signals', 'price_now', price_now, {'ticker_name': ticker})

for ticker, signal, price_now, score in signals[['Акция', 'Сигнал', 'Актуальная цена', 'Score']].itertuples(index=False):
    if ticker in table_of_signals.index:
        db.update_table('signals', 'price_now', price_now, {'ticker_name': ticker})
        db.update_table('signals', 'score', score, {'ticker_name': ticker})
        if table_of_signals.loc[ticker, 'signal'] != signal:
            db.insert_into('history', columns_history, f"""({ticker},
                           {table_of_signals.loc[ticker, 'signal']},
                           {table_of_signals.loc[ticker, 'start_date']},
                           '{datetime.date().isoformat()}',
                           {table_of_signals.loc[ticker, 'start_price']}
                           {price_now},
                           {round((price_now - table_of_signals.loc[ticker, 'start_price']) / table_of_signals.loc[ticker, 'start_price'], 2)})""")

            db.delete_from('signals', {'ticker_name': ticker})
            db.insert_into('signals', columns_signals, f"('{ticker}', '{datetime.now().date().isoformat()}', '{signal}', {price_now}, {price_now}, {score})")
    else:
        db.insert_into('signals', columns_signals, f"('{ticker}', '{datetime.now().date().isoformat()}', '{signal}', {price_now}, {price_now}, {score})")

C:\Users\Danii\AppData\Local\Temp\ipykernel_13132\3345615752.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  table_of_signals = pd.read_sql_query("""SELECT * FROM signals""", db.conn).set_index('ticker_name')


In [23]:
signals_show = pd.read_sql_query("""SELECT * FROM signals""", db.conn, parse_dates='start_date').set_index('ticker_name')
signals_show

C:\Users\Danii\AppData\Local\Temp\ipykernel_13132\2744227270.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  signals_show = pd.read_sql_query("""SELECT * FROM signals""", db.conn, parse_dates='start_date').set_index('ticker_name')


,start_date,signal,start_price,price_now,score
ticker_name,,,,,
IRAO,2026-03-15,buy,3.18,3.21,6.40
NVTK,2026-03-15,sell,1395.80,1421.50,0.47
HNFG,2026-03-15,buy,478.40,476.00,8.74
RENI,2026-03-15,buy,92.84,91.66,8.25
MDMG,2026-03-15,buy,1408.40,1401.40,7.15
MOEX,2026-03-15,buy,174.90,173.46,6.97
UPRO,2026-03-19,buy,1.49,1.49,6.70
CNTLP,2026-03-15,buy,5.97,5.97,6.65
IVAT,2026-03-19,buy,152.20,152.20,6.43


In [24]:
history_show = pd.read_sql_query("""SELECT * FROM history""", db.conn).set_index('ticker_name')
history_show


C:\Users\Danii\AppData\Local\Temp\ipykernel_13132\752047079.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  history_show = pd.read_sql_query("""SELECT * FROM history""", db.conn).set_index('ticker_name')


,signal,start_date,end_date,start_price,end_price,delta
ticker_name,,,,,,
CNTLP,sell,2026-02-28,2026-03-15,6.20,5.97,-3.80
IRAO,sell,2026-02-28,2026-03-15,3.31,3.18,-4.20
SBER,buy,2026-02-27,2026-03-15,302.77,317.47,4.63
RNFT,buy,2026-02-25,2026-03-15,111.17,142.20,21.82
TATN,buy,2026-02-24,2026-03-15,549.98,636.70,13.62
BANE,buy,2026-02-25,2026-03-15,1430.37,1727.50,17.20
FIXR,buy,2026-02-14,2026-03-15,0.56,0.61,8.77
MOEX,sell,2026-02-05,2026-03-15,183.73,174.90,-5.05
UPRO,sell,2026-02-05,2026-03-15,1.64,1.54,-6.71
